In [ ]:
#!python -m pip install pyspellchecker

#!python -m --upgrade pip

#!python -m pip install fasttext-wheel
!python -m pip install symspellpy



In [ ]:
#import nltk
#nltk.download()

In [1]:
import pandas as pd
import numpy as np
import pysrt
import aspose.zip as az
import zipfile
import os
import unidecode
import glob
import re as re
import nltk
from nltk.corpus import stopwords
from langdetect import detect
import sklearn


# ouverture

In [40]:
def ouverture_fichier(nom,encoding='utf-8'):
    with open(nom, 'r', encoding=encoding) as fichier:
        contenu = fichier.read()
    return(contenu)

In [ ]:
contenu=ouverture_fichier("S_netoyller\\Angel\Angel S1E01 - FR - DVDRiP by mstoll.srt")

# traitemant mono fichier

regex a utiliser (?!\d+\s+\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}).+

In [3]:
def enlever_marqueur_temps(episode):
    # Enlever les lignes correspondant aux timecodes et indices
    result = re.sub(r"\d+\s+\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}\s*", '', episode)

    # Afficher le résultat
    return(result)

## retirer la mise en page 

regex a utiliser r"\n*"

In [9]:
def elever_retours_ligne(episode):
    # Enlever les lignes correspondant aux timecodes et indices
    result = re.sub(r"\n*", '', episode)
    # Afficher le résultat
    return(result)


## virer les mots inutil WIP

In [19]:
def remove_punctuation_attached_to_characters(text):
    # Utilisation de l'expression régulière pour supprimer les points, virgules et ellipses collés à des caractères
    return re.sub(r'(?<=\w)[.,](?=\w)|(?<=\w)\.\.(?=\w)', '', text)

In [20]:
def remove_punctuation_from_list(text_list):
    # Expression régulière pour supprimer toute ponctuation
    return [re.sub(r'[^\w\s]', '', text) for text in text_list]

In [21]:
def remove_punctuation(text):
    # Expression régulière pour supprimer toute ponctuation sauf l'apostrophe (')
    return re.sub(r"[^\w\s']", ' ', text)

In [22]:
def remove_i_tags(text):
    # Expression régulière pour supprimer la balise <i>
    return re.sub(r'<i>', '</i>', text)

In [23]:
def replace_apostrophe_with_space(text):
    # Expression régulière pour remplacer l'apostrophe par un espace
    return re.sub(r"'", ' ', text)

# gestion dossier 

In [ ]:
import os
import shutil

def extraire_documents(name):
    source_dir="sous-titres\\"+name
    destination_dir="S_netoyller\\"+name
    # Vérifier si le dossier source existe
    if not os.path.exists(source_dir):
        print(f"Le dossier source {source_dir} n'existe pas.")
        return

    # Créer le dossier de destination s'il n'existe pas déjà
    if not os.path.exists(destination_dir):
        os.makedirs(destination_dir)

    # Parcourir tous les fichiers du dossier source
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            # Construire le chemin complet du fichier
            source_file_path = os.path.join(root, file)
            destination_file_path = os.path.join(destination_dir, file)

            # Copier le fichier vers le dossier de destination
            shutil.copy2(source_file_path, destination_file_path)
            print(f"Fichier {file} copié vers {destination_dir}")
extraire_documents("angel")


In [18]:
def ouverturezip_saison(name, saison_name):
    # Spécifiez le chemin de l'archive ZIP et du répertoire de destination
    saison_name=saison_name+".zip"
    zip_path = os.path.join("sous-titres", name, saison_name,)
    extract_path = os.path.join("S_netoyller", name)

    # Vérifier si le fichier ZIP existe
    if not os.path.exists(zip_path):
        print(f"Le fichier {zip_path} n'existe pas.")
        return
    # Ouvrir et extraire l'archive ZIP
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(extract_path)
        print(f"Les fichiers ont été extraits dans le répertoire : {extract_path}")




In [ ]:
# Appeler la fonction
extraire_documents("angel")
ouverturezip_saison("angel", "angels01VF")

In [25]:
import os
import zipfile
import time

def unzip_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.zip'):
                file_path = os.path.join(root, file)
                try:
                    with zipfile.ZipFile(file_path, 'r') as zip_ref:
                        zip_ref.extractall(root)
                        print('Extraction de', file_path)
                    time.sleep(0.1)
                    os.remove(file_path)
                    print('Suppression de', file_path)
                except zipfile.BadZipFile:
                    print(f"Le fichier ZIP est corrompu et ne peut pas être ouvert : {file_path}")
                    try:
                        os.remove(file_path)
                        print(f"Suppression du fichier ZIP corrompu : {file_path}")
                    except Exception as e:
                        print(f"Impossible de supprimer le fichier corrompu : {file_path}. Erreur : {e}")
                except PermissionError as e:
                    print(f"Erreur de permission lors de la suppression de {file_path}: {e}")

directory = "sous-titres"
unzip_files(directory)

In [ ]:


# Chemin vers le dossier que vous voulez explorer
dossier = 'sous-titres/24'

# Utilisation de glob pour obtenir tous les fichiers .srt dans le dossier
fichiers_srt = glob.glob(os.path.join(dossier, "*.srt"))
sous_dossiers = [f for f in os.listdir(dossier) if os.path.isdir(os.path.join(dossier, f))]
print(sous_dossiers)

In [ ]:

print(ouverture_fichier(fichiers_srt[5]))

# Code trétement zip V1

In [34]:
# En ou fr
def adaptation(langue):
    if langue=='fr':
        return('french')
    return('english')

In [42]:

def ajouter_csv(nom, mots_filtrés):
    # Le chemin du fichier CSV
    chemin_csv = os.path.join('resultat1', nom + '.csv')

    # Vérifiez si le fichier CSV existe
    try:
        # Lire le fichier CSV existant
        df = pd.read_csv(chemin_csv, header=None)
        # Si le fichier est vide, df sera un DataFrame vide
        if df.empty:
            df = pd.DataFrame([mots_filtrés])  # Initialiser avec les mots filtrés
        else:
            # Ajouter les nouveaux mots à la première (et seule) ligne
            df.iloc[0] = df.iloc[0].tolist() + mots_filtrés
    except pd.errors.EmptyDataError:
        # Si le fichier est vide, initialiser un DataFrame avec les mots filtrés
        df = pd.DataFrame([mots_filtrés])

    # Sauvegarder le DataFrame mis à jour dans le fichier CSV
    df.to_csv(chemin_csv, index=False, header=False)



In [61]:
def parcourir_toute_archive(dossier='sous-titres',nom="d"):
    # Récupérer le nom de tous les sous-dossiers dans le dossier courant
    # 'Liste_nom_serie' contient une liste de tous les sous-dossiers
    Liste_nom_serie = [f for f in os.listdir(dossier) if os.path.isdir(os.path.join(dossier, f))]
    nltk.download('stopwords') 
    # Convertir la liste en un tableau NumPy pour des manipulations futures
    Liste_nom_serie = np.array(Liste_nom_serie)
    stop_wordsFr = set(stopwords.words('french'))
    stop_wordsEn = set(stopwords.words('english'))
    # Boucle à travers chaque sous-dossier
    for i in range(Liste_nom_serie.size):
        # Construire le chemin vers le sous-dossier courant
        courent = dossier + "/" + Liste_nom_serie[i]
        if courent==("sous-titres/"+Liste_nom_serie[i]):
            nom=Liste_nom_serie[i]
        # Récupérer tous les sous-dossiers dans le sous-dossier courant
        # 'verifdossier' contient une liste des sous-dossiers du sous-dossier courant
        verifdossier = np.array([f for f in os.listdir(courent) if os.path.isdir(os.path.join(courent, f))])
        
        # Vérification : si le sous-dossier courant contient encore des sous-dossiers 
        if verifdossier.size != 0:
            # Appel récursif de la fonction pour parcourir plus profondément les sous-dossiers
            parcourir_toute_archive(courent,nom=nom)
        # Récupérer tous les fichiers avec l'extension .srt dans le dossier courant
        fichiers_srt = np.array(glob.glob(os.path.join(dossier, "*.srt")))
        print(fichiers_srt)
        # Boucle à travers chaque fichier .srt trouvé
        val=[]
        for i in range(fichiers_srt.size):
            contenu=ouverture_fichier(fichiers_srt[i])
            contenu=enlever_marqueur_temps(contenu)
            contenu=elever_retours_ligne(contenu)
            contenu=enlever_ponctuation(contenu)
            langue = detect(contenu)
            langue=adaptation(langue)
            if (langue=='french'):
                mots_filtrés = [mot for mot in contenu.split() if mot.lower() not in stop_wordsFr]
                #print("1")
            else:
                mots_filtrés = [mot for mot in contenu.split() if mot.lower() not in stop_wordsEn]
            val=val+mots_filtrés
        print(nom)
        #with open("resultat1/"+nom+".txt", "w") as fichier:
         #   fichier.write(str(val))
          #  print(len(val))        





In [ ]:
#parcourir_toute_archive()

In [ ]:
dossier2='sous-titres'
Liste_nom_serie = [f for f in os.listdir(dossier2) if os.path.isdir(os.path.join(dossier2, f))]
Liste_nom_serie=np.array(Liste_nom_serie)
print(Liste_nom_serie.size)

# Code trétement zip V2

In [31]:
nltk.download('stopwords') 
# Convertir la liste en un tableau NumPy pour des manipulations futures
stop_wordsFr = set(stopwords.words('french'))
stop_wordsEn = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\natha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [30]:
def new_fichier(nom, french, ajouter, ecart=-3):
    if french:
        result = nom[:ecart]
        chemin_fichier = 'train2/' + result + '.txt'
    else:
        chemin_fichier = 'train2/' + nom + '.txt'
    
    # Créer un fichier si nécessaire
    os.makedirs(os.path.dirname(chemin_fichier), exist_ok=True)
    
    with open(chemin_fichier, 'a', encoding='utf-8') as fichier:
        fichier.write(" ".join(ajouter) + " ")
    
    return nom


In [32]:
fichiers_srt= np.array(glob.glob(os.path.join("train", "*.txt")))
len(fichiers_srt)
fichiers_srt[220]

np.str_('train\\thevampirediaries_VO.txt')

In [41]:
fichiers_srt = np.array(glob.glob(os.path.join("clean_subtitles", "*.txt")))
#print(fichiers_srt)
# Boucle à travers chaque fichier .srt trouvé
val=[]
for i in range(fichiers_srt.size):
    contenu=ouverture_fichier(fichiers_srt[i])
    contenu=enlever_marqueur_temps(contenu)
    contenu=elever_retours_ligne(contenu)
    langue = detect(contenu)
    langue=adaptation(langue)
    print(fichiers_srt[i])
    if (langue=='french'):
        interait=fichiers_srt[i]
        interait=interait[6:]
        mots_filtrés = [mot for mot in contenu.split() if mot.lower() not in stop_wordsFr]
        f_nom=new_fichier(interait,True,mots_filtrés,ecart=-6)
    else:
        mots_filtrés = [mot for mot in contenu.split() if mot.lower() not in stop_wordsEn]
        new_fichier(f_nom,True,mots_filtrés,ecart=-6)

clean_subtitles\24_VF.txt
clean_subtitles\24_VO.txt
clean_subtitles\90210_VF.txt
clean_subtitles\90210_VO.txt
clean_subtitles\alias_VF.txt
clean_subtitles\alias_VO.txt
clean_subtitles\angel_VF.txt
clean_subtitles\angel_VO.txt
clean_subtitles\battlestargalactica_VF.txt
clean_subtitles\battlestargalactica_VO.txt
clean_subtitles\betteroffted_VF.txt
clean_subtitles\betteroffted_VO.txt
clean_subtitles\bionicwoman_VF.txt
clean_subtitles\bionicwoman_VO.txt
clean_subtitles\blade_VO.txt
clean_subtitles\bloodties_VF.txt
clean_subtitles\bloodties_VO.txt
clean_subtitles\bones_VF.txt
clean_subtitles\bones_VO.txt
clean_subtitles\breakingbad_VF.txt
clean_subtitles\breakingbad_VO.txt
clean_subtitles\buffy_VF.txt
clean_subtitles\buffy_VO.txt
clean_subtitles\burnnotice_VF.txt
clean_subtitles\burnnotice_VO.txt
clean_subtitles\californication_VF.txt
clean_subtitles\californication_VO.txt
clean_subtitles\caprica_VF.txt
clean_subtitles\caprica_VO.txt
clean_subtitles\charmed_VF.txt
clean_subtitles\charmed_VO

In [ ]:
i=219
while (i!=fichiers_srt.size):
    contenu=ouverture_fichier(fichiers_srt[i])
    contenu=enlever_marqueur_temps(contenu)
    contenu=elever_retours_ligne(contenu)
    langue = detect(contenu)
    langue=adaptation(langue)
    print(fichiers_srt[i])
    if (langue=='french'):
        interait=fichiers_srt[i]
        interait=interait[6:]
        mots_filtrés = [mot for mot in contenu.split() if mot.lower() not in stop_wordsFr]
        f_nom=new_fichier(interait,True,mots_filtrés,ecart=-6)
    else:
        mots_filtrés = [mot for mot in contenu.split() if mot.lower() not in stop_wordsEn]
        new_fichier(f_nom,True,mots_filtrés,ecart=-6)
    i+=1

In [ ]:
fichiers_srt = np.array(glob.glob(os.path.join("train", "*.txt")))
print(fichiers_srt)

In [38]:
# Ouvrir un fichier en mode ajout et y écrire du texte
b=new_fichier('non_vf',True,'ajouter')
new_fichier(b,False,'add')


# 2 eme partie du netoyage

In [13]:
nltk.download('stopwords') 
# Convertir la liste en un tableau NumPy pour des manipulations futures
stop_wordsFr = set(stopwords.words('french'))
stop_wordsEn = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\natha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
mots_inutiles = [
    # Articles définis et indéfinis
    "le", "la", "les", "un", "une", "des", "du", "de", "de la", "de l'", "au", "aux",

    # Conjonctions
    "et", "ou", "mais", "donc", "or", "ni", "car",

    # Prépositions
    "à", "de", "en", "dans", "sur", "sous", "par", "pour", "avec", "sans", "chez", "vers", 
    "entre", "devant", "derrière",

    # Pronoms personnels, relatifs, et démonstratifs
    "je", "tu", "il", "elle", "nous", "vous", "ils", "elles", "me", "te", "se", "nous", "vous", 
    "leur", "le", "la", "les", "lui", "qui", "que", "quoi", "dont", "où", "ce", "cet", "cette", 
    "ces", "ça", "cela", "celui", "celle", "ceux", "celles", "jai", "tout", "tous", "ces", "ses", "cest",

    # Verbes auxiliaires
    "être", "suis", "es", "est", "sommes", "êtes", "sont", 
    "avoir", "ai", "as", "a", "avons", "avez", "ont",

    # Adverbes et particules courants
    "ne", "pas", "plus", "ni", "si", "oui", "non", "aussi", "bien", "encore", "déjà", 
    "toujours", "jamais",

    # Autres mots communs
    "tout", "tous", "toute", "toutes", "même", "autre", "autres", "quelque", "quelques", 
    "plusieurs", "chaque", "chacun", "chacune", "beaucoup", "trop", "peu", "très", "moins", "plus",'oui','non'
]


In [14]:
def add_space_between_word_and_uppercase(text):
    # Expression régulière pour détecter un mot suivi d'une lettre majuscule sans espace
    pattern =  r'([a-z])([A-Z])'
    
    # Remplacer les correspondances par un mot suivi d'un espace et la majuscule
    corrected_text = re.sub(pattern, r'\1 \2', text)
    
    return corrected_text
def traiter_mot(mot):
    return unidecode.unidecode(mot).lower()

### trop lente

In [48]:
# Fonction pour corriger les mots après segmentation selon la langue détectée
def corriger_mots_segmentes(mots):
    from wordsegment import load, segment
    from spellchecker import SpellChecker
    mots_corriges = []
    for mot in mots:
        try:
            langue = detect(mot)
            # Corriger selon la langue détectée
            if langue == 'fr':
                correction = spell_fr.correction(mot)
            elif langue == 'en':
                correction = spell_en.correction(mot)
            else:
                correction = mot  # Ne pas corriger si la langue n'est ni français ni anglais
        except:
            correction = mot  # Si la détection échoue, conserver le mot original
        
        # Ajouter la correction (ou le mot original s'il n'y a pas de correction)
        mots_corriges.append(correction if correction else mot)
    
    return mots_corriges

# Fonction pour segmenter et corriger les mots collés
def separer_et_corriger(liste_mots):
    from wordsegment import load, segment
    from spellchecker import SpellChecker
    resultats = []
    for mot in liste_mots:
        # Segmenter le mot collé
        mots_segmentes = segment(mot)
        # Corriger les mots segmentés
        mots_corriges = corriger_mots_segmentes(mots_segmentes)
        resultats.extend(mots_corriges)
    return resultats





In [49]:
def corecteur_séparateur(mots_colles):
    from wordsegment import load, segment
    from spellchecker import SpellChecker
    # Charger le modèle WordSegment
    load()
    spell_fr = SpellChecker(language='fr')
    spell_en = SpellChecker(language='en')
    # Appliquer la fonction pour segmenter et corriger
    mots_corriges = separer_et_corriger(mots_colles, n_jobs=-1)
    return(mots_corriges)

### autre

In [47]:
test=ouverture_fichier('train2/24_.txt')
print(len(test))
test=remove_punctuation_attached_to_characters(test)
test=remove_i_tags(test)
test= replace_apostrophe_with_space(test)
test=remove_punctuation(test)
test=add_space_between_word_and_uppercase(test)
mots_filtrés = [mot for mot in test.split() if mot.lower() not in stop_wordsFr]
mots_filtrés= [mot for mot in mots_filtrés if mot.lower() not in stop_wordsEn]
mots_filtrés = list(filter(lambda mot: len(mot) > 3, mots_filtrés))
mots_filtrés= [mot for mot in mots_filtrés if mot.lower() not in mots_inutiles]
print(len(mots_filtrés))
print(mots_filtrés[1000:2000])

4649622
588114
['pouvez', 'soutenir', 'trouverais', 'secrétaire', 'état', 'pourra', 'écouterai', 'mécontentement', 'excusez', 'présidente', 'woods', 'sécurité', 'territoire', 'besoin', 'propos', 'choses', 'attendre', 'madame', 'accord', 'continuerons', 'veux', 'rapport', 'détails', 'agit', 'potentielle', 'fuite', 'domestique', 'briefer', 'chefs', 'état', 'major', 'attendent', 'enquêté', 'dernières', 'semaines', 'vols', 'technologies', 'implications', 'hommes', 'vols', 'enlevés', 'michael', 'latham', 'ingénieur', 'chargé', 'protéger', 'infrastructures', 'importantes', 'dites', 'faire', 'hommes', 'forcé', 'latham', 'créer', 'objet', 'pouvant', 'pirater', 'pare', 'pare', 'protège', 'systèmes', 'nationaux', 'energie', 'machines', 'transport', 'rapporté', 'irrégularités', 'trafic', 'aérien', 'voulez', 'vols', 'commerciaux', 'danger', 'dois', 'donner', 'ordre', 'cesser', 'trafic', 'aérien', 'peur', 'simple', 'moment', 'vols', 'atterrir', 'faudrait', 'heures', 'minimum', 'créer', 'panique', '

### teste fast text

In [ ]:
import fasttext
from wordsegment import load, segment
from symspellpy.symspellpy import SymSpell, Verbosity

# Charger le modèle de détection de langue FastText
fasttext_model = fasttext.load_model('lid.176.bin')  # Télécharger depuis https://fasttext.cc/docs/en/language-identification.html

# Charger le modèle de segmentation WordSegment
load()

# Initialiser SymSpell pour la correction orthographique
symspell_fr = SymSpell()
symspell_en = SymSpell()

# Charger les dictionnaires français et anglais pour SymSpell
symspell_fr.load_dictionary('french_dictionary.txt', 0, 1)
symspell_en.load_dictionary('english_dictionary.txt', 0, 1)

# Fonction pour corriger un mot selon la langue détectée
def corriger_mot(mot, langue):
    if langue == 'fr':
        suggestions = symspell_fr.lookup(mot, Verbosity.CLOSEST, max_edit_distance=2)
    elif langue == 'en':
        suggestions = symspell_en.lookup(mot, Verbosity.CLOSEST, max_edit_distance=2)
    else:
        return mot  # Si la langue n'est ni français ni anglais, ne pas corriger
    return suggestions[0].term if suggestions else mot

# Fonction pour détecter la langue avec FastText
def detecter_langue(mot):
    pred = fasttext_model.predict(mot)
    langue = pred[0][0].split("__label__")[1]
    return langue

# Fonction pour segmenter, détecter la langue et corriger les mots collés
def separer_corriger_mots(liste_mots):
    resultats = []
    for mot in liste_mots:
        # Segmenter le mot collé en sous-mots
        mots_segmentes = segment(mot)
        mots_corriges = []
        for sous_mot in mots_segmentes:
            langue = detecter_langue(sous_mot)
            mot_corrige = corriger_mot(sous_mot, langue)
            mots_corriges.append(mot_corrige)
        resultats.extend(mots_corriges)
    return resultats

# Exemple de texte avec des mots collés
mots_colles = ['coursinuitils', 'ourhouseisbig', 'nôtresiipuis']

# Appliquer la fonction de séparation et correction
mots_corriges = separer_corriger_mots(mots_colles)

# Afficher les résultats
print(mots_corriges)


In [ ]:
v=0
for i in range(len(mots_filtrés)):
    if mots_filtrés[i]=='Losti':
        v+=1
print(v)